# 18 从零实现 softmax 回归

本节目标：展平图像、手写线性分类器、手写交叉熵、计算准确率，并完成一个小训练循环。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset

try:
    from torchvision import datasets, transforms
    HAS_TORCHVISION = True
except Exception as exc:
    HAS_TORCHVISION = False
    TORCHVISION_ERROR = exc


def load_fashion_mnist(train_limit=1024, test_limit=512, batch_size=256):
    data_root = Path("../data")
    if HAS_TORCHVISION:
        try:
            transform = transforms.ToTensor()
            train_full = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
            test_full = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)
            train_data = Subset(train_full, range(min(train_limit, len(train_full))))
            test_data = Subset(test_full, range(min(test_limit, len(test_full))))
            source = "Fashion-MNIST"
        except Exception as exc:
            print("Fashion-MNIST 下载或读取失败，改用同 shape 的随机数据继续练习:", repr(exc))
            train_images = torch.rand(train_limit, 1, 28, 28)
            train_labels = torch.randint(0, 10, (train_limit,))
            test_images = torch.rand(test_limit, 1, 28, 28)
            test_labels = torch.randint(0, 10, (test_limit,))
            train_data = TensorDataset(train_images, train_labels)
            test_data = TensorDataset(test_images, test_labels)
            source = "synthetic fallback"
    else:
        print("torchvision 不可用，改用同 shape 的随机数据继续练习:", repr(TORCHVISION_ERROR))
        train_images = torch.rand(train_limit, 1, 28, 28)
        train_labels = torch.randint(0, 10, (train_limit,))
        test_images = torch.rand(test_limit, 1, 28, 28)
        test_labels = torch.randint(0, 10, (test_limit,))
        train_data = TensorDataset(train_images, train_labels)
        test_data = TensorDataset(test_images, test_labels)
        source = "synthetic fallback"

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, source

import pandas as pd

torch.manual_seed(42)
torch.set_printoptions(precision=4)

train_loader, test_loader, source = load_fashion_mnist(train_limit=1024, test_limit=512, batch_size=256)
print("数据来源:", source)

num_inputs = 28 * 28
num_outputs = 10
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)


def net(X):
    return X.reshape((-1, num_inputs)) @ W + b


def cross_entropy(logits, y):
    log_probs = torch.log_softmax(logits, dim=1)
    return -log_probs[torch.arange(len(y)), y].mean()


def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def evaluate(data_loader):
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            logits = net(X_batch)
            loss = cross_entropy(logits, y_batch)
            batch_size = len(y_batch)
            total_loss += loss.item() * batch_size
            total_acc += accuracy(logits, y_batch) * batch_size
            total_count += batch_size
    return total_loss / total_count, total_acc / total_count

lr = 0.1
num_epochs = 3
logs = []

for epoch in range(num_epochs):
    total_loss = 0.0
    total_acc = 0.0
    total_count = 0
    for X_batch, y_batch in train_loader:
        logits = net(X_batch)
        loss = cross_entropy(logits, y_batch)
        loss.backward()
        with torch.no_grad():
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_()
            b.grad.zero_()
        batch_size = len(y_batch)
        total_loss += loss.item() * batch_size
        total_acc += accuracy(logits, y_batch) * batch_size
        total_count += batch_size

    test_loss, test_acc = evaluate(test_loader)
    row = {
        "epoch": epoch + 1,
        "train_loss": total_loss / total_count,
        "train_acc": total_acc / total_count,
        "test_acc": test_acc,
    }
    logs.append(row)
    print(row)

log_table = pd.DataFrame(logs)
print("\n训练日志:")
print(log_table)

## 小结

从零实现 softmax 回归时，我们自己写了图像展平、线性分类器、交叉熵、准确率和参数更新。它能帮助理解分类模型的训练细节。